# Geohash-based Environment Classification Pipeline

Optimized for 40M+ POIs with:
- Vectorized geohash encoding
- Parallel processing
- Chunked memory-efficient processing
- NumPy optimizations

In [ ]:
import pandas as pd
import numpy as np
import pygeohash as pgh
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import joblib
from joblib import Parallel, delayed
from tqdm import tqdm
import warnings
import gc
warnings.filterwarnings('ignore')

# For even faster geohash encoding
try:
    import geohash2 as gh2
    USE_GEOHASH2 = True
except ImportError:
    USE_GEOHASH2 = False
    print("Install geohash2 for faster encoding: pip install geohash2")

In [ ]:
class FastUnsupervisedEnvironmentClassifier:
    """
    Environment Classification focused on Urban-Rural spectrum
    Excludes transportation/highway POIs from analysis
    """
    
    def __init__(self, geohash_precision=6, n_clusters=7, n_jobs=-1, chunk_size=500000):
        self.precision = geohash_precision
        self.n_clusters = n_clusters
        self.n_jobs = n_jobs
        self.chunk_size = chunk_size
        
        # Categories for analysis (transportation excluded from classification)
        self.poi_categories = [
            'transportation', 'public_services', 'business', 
            'tourism', 'infrastructure', 'residential', 'other'
        ]
        self.category_to_idx = {cat: i for i, cat in enumerate(self.poi_categories)}
        
        # Exclude transportation from classification features
        self.active_categories = ['public_services', 'business', 'tourism', 
                                   'infrastructure', 'residential', 'other']
        self.active_indices = [1, 2, 3, 4, 5, 6]  # Skip index 0 (transportation)
        
        self.scaler = StandardScaler()
        self.clustering_model = None
        self.cell_clusters = {}
        self.cluster_names = {}
        
    def _encode_geohash_batch(self, lats, lons):
        """Vectorized geohash encoding"""
        if USE_GEOHASH2:
            return [gh2.encode(lat, lon, self.precision) for lat, lon in zip(lats, lons)]
        else:
            return [pgh.encode(lat, lon, precision=self.precision) for lat, lon in zip(lats, lons)]
    
    def _encode_geohash_parallel(self, df):
        """Parallel geohash encoding for large datasets"""
        n_chunks = max(1, len(df) // self.chunk_size)
        chunks = np.array_split(df.index, n_chunks)
        
        def process_chunk(indices):
            chunk_df = df.loc[indices]
            return self._encode_geohash_batch(
                chunk_df['latitude'].values,
                chunk_df['longitude'].values
            )
        
        results = Parallel(n_jobs=self.n_jobs, backend='threading')(
            delayed(process_chunk)(chunk) for chunk in tqdm(chunks, desc="Encoding geohashes")
        )
        
        return [gh for result in results for gh in result]
    
    def _get_neighbor_geohashes(self, geohash):
        """Get all 8 neighboring geohash cells"""
        neighbors = []
        directions = ['top', 'bottom', 'right', 'left', 
                     'topleft', 'topright', 'bottomleft', 'bottomright']
        
        for direction in directions:
            try:
                neighbor = pgh.get_adjacent(geohash, direction)
                neighbors.append(neighbor)
            except:
                pass
        
        return neighbors
    
    def aggregate_poi_by_geohash(self, poi_df):
        """Aggregate POIs excluding transportation"""
        print("Aggregating POIs by geohash (excluding transportation)...")
        
        poi_df = poi_df.copy()
        geohash_col = f'geohash_{self.precision}'
        
        # Fast parallel geohash encoding
        print(f"Encoding {len(poi_df):,} POIs...")
        poi_df[geohash_col] = self._encode_geohash_parallel(poi_df)
        
        # Convert category to numeric
        poi_df['category_idx'] = poi_df['category'].map(self.category_to_idx).fillna(6).astype(int)
        
        # Get POI counts per cell and category
        print("Pre-aggregating POI counts...")
        poi_counts = poi_df.groupby([geohash_col, 'category_idx']).size().unstack(fill_value=0)
        
        # Ensure all categories exist
        for i in range(len(self.poi_categories)):
            if i not in poi_counts.columns:
                poi_counts[i] = 0
        poi_counts = poi_counts[sorted(poi_counts.columns)]
        
        # Get counts excluding transportation (index 0)
        poi_counts_no_transport = poi_counts[self.active_indices]
        total_pois_no_transport = poi_counts_no_transport.sum(axis=1)
        
        # Build neighbor lookup
        print("Building neighbor lookup...")
        unique_cells = poi_counts.index.tolist()
        cell_set = set(unique_cells)
        
        neighbor_counts = {}
        for cell in tqdm(unique_cells, desc="Computing neighbor features"):
            neighbors = self._get_neighbor_geohashes(cell)
            neighbor_sum = np.zeros(len(self.active_indices))
            for n in neighbors:
                if n in cell_set:
                    neighbor_sum += poi_counts_no_transport.loc[n].values
            neighbor_counts[cell] = neighbor_sum
        
        # Extract features
        print("Extracting features...")
        cell_data = []
        
        for cell in tqdm(unique_cells, desc="Building feature vectors"):
            counts = poi_counts_no_transport.loc[cell].values.astype(float)
            total = total_pois_no_transport[cell]
            neighbor_sum = neighbor_counts[cell]
            neighbor_total = neighbor_sum.sum()
            
            center = pgh.decode(cell)
            features = self._extract_urban_rural_features(counts, total, neighbor_sum, neighbor_total)
            
            cell_data.append({
                'geohash': cell,
                'latitude': center[0],
                'longitude': center[1],
                'features': features,
                'total_pois': total,
                'raw_counts': counts
            })
        
        print(f"Created features for {len(cell_data):,} cells")
        return pd.DataFrame(cell_data)
    
    def _extract_urban_rural_features(self, poi_counts, total_pois, neighbor_counts, neighbor_total):
        """Extract features focused on urban-rural classification"""
        features = []
        
        # Indices in active categories: public_services=0, business=1, tourism=2, 
        # infrastructure=3, residential=4, other=5
        
        # 1. POI Type Distribution (6 features) - no transportation
        poi_ratios = poi_counts / (total_pois + 1e-10)
        features.extend(poi_ratios)
        
        # 2. Total POI Density - key urban indicator
        features.append(np.log1p(total_pois))
        
        # 3. Diversity Metrics (3 features)
        if total_pois > 0:
            # Shannon entropy
            p = poi_counts / total_pois
            p = p[p > 0]
            entropy = -np.sum(p * np.log(p)) if len(p) > 0 else 0
            features.append(entropy)
            
            # Dominant ratio
            features.append(np.max(poi_counts) / total_pois)
            
            # Category coverage
            features.append(np.sum(poi_counts > 0) / len(self.active_categories))
        else:
            features.extend([0, 0, 0])
        
        # 4. Urban Activity Indicators (5 features)
        if total_pois > 0:
            # Commercial intensity (business + tourism)
            commercial = (poi_counts[1] + poi_counts[2]) / total_pois
            features.append(commercial)
            
            # Service accessibility (public_services)
            features.append(poi_counts[0] / total_pois)
            
            # Residential density
            features.append(poi_counts[4] / total_pois)
            
            # Business to residential ratio
            features.append(np.log1p((poi_counts[1] + 1) / (poi_counts[4] + 1)))
            
            # Mixed-use score (multiple categories present)
            active_types = np.sum(poi_counts > 0)
            features.append(active_types / len(self.active_categories))
        else:
            features.extend([0, 0, 0, 0, 0])
        
        # 5. Neighborhood Context (6 features)
        neighbor_ratios = neighbor_counts / (neighbor_total + 1e-10)
        features.extend(neighbor_ratios)
        
        # 6. Spatial Contrast (6 features)
        contrast = poi_ratios - neighbor_ratios
        features.extend(contrast)
        
        # 7. Urban-Rural Indicators (4 features)
        # Urban score: high density + commercial + services
        urban_score = np.log1p(total_pois) * 0.3
        if total_pois > 0:
            urban_score += (poi_counts[0] + poi_counts[1] + poi_counts[2]) / total_pois * 0.7
        features.append(urban_score)
        
        # Residential character
        residential_char = poi_counts[4] / (total_pois + 1e-10)
        features.append(residential_char)
        
        # Industrial character
        industrial_char = poi_counts[3] / (total_pois + 1e-10)
        features.append(industrial_char)
        
        # Sparsity indicator (inverse density)
        features.append(1 / (total_pois + 1))
        
        return np.array(features)
    
    def find_optimal_clusters(self, cell_df, max_clusters=15):
        """Find optimal clusters"""
        print("\nFinding optimal number of clusters...")
        
        X = np.vstack(cell_df['features'].values)
        X_scaled = self.scaler.fit_transform(X)
        
        if len(X_scaled) > 10000:
            sample_idx = np.random.choice(len(X_scaled), 10000, replace=False)
            X_sample = X_scaled[sample_idx]
        else:
            X_sample = X_scaled
        
        silhouette_scores = []
        K_range = range(3, max_clusters + 1)
        
        for k in tqdm(K_range, desc="Testing clusters"):
            kmeans = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=1024, n_init=3)
            labels = kmeans.fit_predict(X_sample)
            silhouette_scores.append(silhouette_score(X_sample, labels))
        
        optimal_k = K_range[np.argmax(silhouette_scores)]
        print(f"Optimal clusters: {optimal_k}")
        
        return optimal_k
    
    def cluster_cells(self, cell_df, method='minibatch_kmeans', n_clusters=None):
        """Cluster cells"""
        if n_clusters is None:
            n_clusters = self.n_clusters
        
        print(f"\nClustering {len(cell_df):,} cells...")
        
        X = np.vstack(cell_df['features'].values)
        X_scaled = self.scaler.fit_transform(X)
        
        if method == 'minibatch_kmeans':
            self.clustering_model = MiniBatchKMeans(
                n_clusters=n_clusters, random_state=42, batch_size=1024, n_init=10
            )
        else:
            self.clustering_model = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        
        labels = self.clustering_model.fit_predict(X_scaled)
        cell_df['cluster'] = labels
        
        self.cell_clusters = dict(zip(cell_df['geohash'], cell_df['cluster']))
        self._analyze_clusters(cell_df)
        
        print(f"Clustering completed! Found {len(np.unique(labels))} clusters")
        return cell_df
    
    def _analyze_clusters(self, cell_df):
        """Analyze and name clusters for urban-rural spectrum"""
        print("\nAnalyzing cluster characteristics...")
        
        for cluster_id in sorted(cell_df['cluster'].unique()):
            if cluster_id == -1:
                continue
            
            cluster_cells = cell_df[cell_df['cluster'] == cluster_id]
            features_matrix = np.vstack(cluster_cells['features'].values)
            avg_features = np.mean(features_matrix, axis=0)
            
            # Get raw counts for distribution
            raw_counts_matrix = np.vstack(cluster_cells['raw_counts'].values)
            avg_raw = np.mean(raw_counts_matrix, axis=0)
            total_raw = avg_raw.sum()
            poi_distribution = avg_raw / (total_raw + 1e-10)
            
            # Get density and other metrics
            avg_density = np.mean(cluster_cells['total_pois'])
            
            cluster_name = self._name_urban_rural_cluster(poi_distribution, avg_density, avg_features)
            self.cluster_names[cluster_id] = cluster_name
            
            print(f"\nCluster {cluster_id}: {cluster_name}")
            print(f"  - Cells: {len(cluster_cells):,}")
            print(f"  - Avg POI density: {avg_density:.1f}")
            print(f"  - Distribution: ", end="")
            for i, cat in enumerate(self.active_categories):
                print(f"{cat[:3]}={poi_distribution[i]:.0%} ", end="")
            print()
    
    def _name_urban_rural_cluster(self, poi_dist, density, features):
        """Name clusters on urban-rural spectrum with unique names"""
        # poi_dist indices: public_services=0, business=1, tourism=2,
        # infrastructure=3, residential=4, other=5

        public_services = poi_dist[0]
        business = poi_dist[1]
        tourism = poi_dist[2]
        infrastructure = poi_dist[3]
        residential = poi_dist[4]
        other = poi_dist[5]

        # Determine density category
        if density > 100:
            density_prefix = "Urban High-Density"
        elif density > 50:
            density_prefix = "Urban"
        elif density > 20:
            density_prefix = "Suburban"
        elif density > 5:
            density_prefix = "Peri-Urban"
        elif density > 1:
            density_prefix = "Rural"
        elif density > 0.1:
            density_prefix = "Sparse Rural"
        else:
            return "Undeveloped"

        # Find dominant category
        categories = [
            ('Services', public_services),
            ('Commercial', business),
            ('Tourism', tourism),
            ('Industrial', infrastructure),
            ('Residential', residential),
            ('Other', other)
        ]

        # Sort by ratio
        sorted_cats = sorted(categories, key=lambda x: x[1], reverse=True)
        dominant_name, dominant_ratio = sorted_cats[0]
        second_name, second_ratio = sorted_cats[1]

        # Very dominant single category (>60%)
        if dominant_ratio > 0.6:
            return f"{density_prefix} {dominant_name}"

        # Strong dominant (>40%) with secondary
        if dominant_ratio > 0.4:
            if second_ratio > 0.2:
                return f"{density_prefix} {dominant_name}-{second_name}"
            return f"{density_prefix} {dominant_name}"

        # Mixed with dominant (>25%)
        if dominant_ratio > 0.25:
            if second_ratio > 0.15:
                return f"{density_prefix} Mixed ({dominant_name}/{second_name})"
            return f"{density_prefix} {dominant_name} Mixed"

        # Highly mixed
        active_cats = sum(1 for _, r in categories if r > 0.1)
        if active_cats >= 3:
            return f"{density_prefix} Mixed-Use"

        return f"{density_prefix} Mixed"
    
    def predict(self, lat, lon):
        """Predict environment type"""
        geohash = pgh.encode(lat, lon, precision=self.precision)
        
        if geohash in self.cell_clusters:
            cluster_id = self.cell_clusters[geohash]
            return {
                'geohash': geohash,
                'cluster_id': cluster_id,
                'environment_type': self.cluster_names.get(cluster_id, f"Cluster {cluster_id}"),
                'method': 'direct_lookup'
            }
        return {
            'geohash': geohash,
            'cluster_id': -1,
            'environment_type': 'Unknown',
            'method': 'not_found'
        }
    
    def predict_batch(self, coords):
        """Batch prediction"""
        return [self.predict(lat, lon) for lat, lon in coords]
    
    def visualize_clusters(self, cell_df, save_path='cluster_map.png'):
        """Visualize clusters"""
        print("\nCreating visualization...")
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        plot_df = cell_df.sample(min(50000, len(cell_df)))
        
        scatter = axes[0].scatter(
            plot_df['longitude'], plot_df['latitude'],
            c=plot_df['cluster'], cmap='RdYlGn_r', alpha=0.6, s=10
        )
        axes[0].set_xlabel('Longitude')
        axes[0].set_ylabel('Latitude')
        axes[0].set_title('Urban-Rural Distribution')
        plt.colorbar(scatter, ax=axes[0], label='Cluster')
        
        cluster_sizes = cell_df['cluster'].value_counts().sort_index()
        colors = plt.cm.RdYlGn_r(np.linspace(0, 1, len(cluster_sizes)))
        axes[1].bar(range(len(cluster_sizes)), cluster_sizes.values, color=colors)
        axes[1].set_xlabel('Cluster ID')
        axes[1].set_ylabel('Number of Cells')
        axes[1].set_title('Cluster Sizes')
        
        plt.tight_layout()
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    def save_model(self, filepath):
        """Save model"""
        joblib.dump({
            'clustering_model': self.clustering_model,
            'scaler': self.scaler,
            'cell_clusters': self.cell_clusters,
            'cluster_names': self.cluster_names,
            'precision': self.precision,
            'n_clusters': self.n_clusters
        }, filepath)
        print(f"Model saved: {filepath}")
    
    def load_model(self, filepath):
        """Load model"""
        data = joblib.load(filepath)
        self.clustering_model = data['clustering_model']
        self.scaler = data['scaler']
        self.cell_clusters = data['cell_clusters']
        self.cluster_names = data['cluster_names']
        self.precision = data['precision']
        self.n_clusters = data['n_clusters']
        print(f"Model loaded: {filepath}")

## Main Pipeline

In [ ]:
def run_pipeline(poi_filepath, n_clusters=7):
    """
    Main pipeline for environment classification
    
    Args:
        poi_filepath: Path to POI CSV with columns: latitude, longitude, category
        n_clusters: Number of environment clusters
    """
    print("="*60)
    print("FAST ENVIRONMENT CLASSIFICATION PIPELINE")
    print("Optimized for 40M+ POIs")
    print("="*60)
    
    # 1. Load data
    print("\n1. Loading POI data...")
    poi_df = pd.read_csv(poi_filepath)
    print(f"   Loaded {len(poi_df):,} POIs")
    print(f"   Categories: {poi_df['category'].nunique()}")
    
    # 2. Initialize classifier
    print("\n2. Initializing classifier...")
    classifier = FastUnsupervisedEnvironmentClassifier(
        geohash_precision=6,
        n_clusters=n_clusters,
        n_jobs=-1,
        chunk_size=500000
    )
    
    # 3. Aggregate by geohash
    print("\n3. Aggregating POIs by geohash...")
    cell_df = classifier.aggregate_poi_by_geohash(poi_df)
    
    # Free memory
    del poi_df
    gc.collect()
    
    # 4. Find optimal clusters (optional)
    print("\n4. Finding optimal clusters...")
    optimal_k = classifier.find_optimal_clusters(cell_df, max_clusters=12)
    
    # 5. Cluster
    print("\n5. Clustering cells...")
    cell_df = classifier.cluster_cells(
        cell_df, 
        method='minibatch_kmeans',
        n_clusters=n_clusters
    )
    
    # 6. Visualize
    print("\n6. Creating visualizations...")
    classifier.visualize_clusters(cell_df)
    
    # 7. Save
    print("\n7. Saving model...")
    classifier.save_model('environment_classifier.pkl')
    
    print("\n" + "="*60)
    print("PIPELINE COMPLETED!")
    print("="*60)
    
    return classifier, cell_df

## Run Pipeline

In [ ]:
# Update this path to your POI data
POI_FILE = 'path/to/your/poi_data.csv'

# Run pipeline
classifier, cell_df = run_pipeline(POI_FILE, n_clusters=7)

## Test Predictions

In [ ]:
# Test coordinates
test_coords = [
    (3.1478, 101.6953),
    (3.1167, 101.6500),
    (3.1412, 101.6931),
]

for lat, lon in test_coords:
    result = classifier.predict(lat, lon)
    print(f"({lat:.4f}, {lon:.4f}) -> {result['environment_type']}")

## Performance Tips for 40M+ POIs

1. **Memory**: Process in chunks, delete intermediate DataFrames
2. **Speed**: Use `geohash2` library, MiniBatchKMeans, parallel processing
3. **Storage**: Save cell_df to parquet for faster reloading
4. **GPU**: Consider cuML for GPU-accelerated clustering

In [ ]:
# Save results to parquet for faster reloading
cell_df_save = cell_df.drop('features', axis=1)
cell_df_save.to_parquet('cell_clusters.parquet', index=False)
print("Saved cell clusters to parquet")